# Лекция:  Метрики качества, валидация и функции потерь в задачах ранжирования

## Содержание

1. [Метрики качества ранжирования](#1-метрики-качества-ранжирования)
   - 1.1 Постановка задачи
   - 1.2 Precision@K и Recall@K
   - 1.3 Average Precision и MAP
   - 1.4 NDCG
   - 1.5 MRR и Hit Rate
   - 1.6 Diversity и Coverage
   - 1.7 Beyond-accuracy метрики
   - 1.8 Как выбирать метрику


2. [Валидация в рекомендательных системах](#2-валидация-в-рекомендательных-системах)
   - 2.1 Почему random split не работает
   - 2.2 Temporal Split
   - 2.3 Протоколы валидации
   - 2.4 Popularity bias
   - 2.5 Exposure bias и IPS-коррекция
   - 2.6 Evaluation на имплицитных данных
   - 2.7 Офлайн vs онлайн метрики
   - 2.8 Типичные ошибки: checklist
   
3. [Learning to Rank: функции потерь и оптимизация](#3-learning-to-rank-функции-потерь-и-оптимизация)
   - 3.1 Парадигмы LTR (Pointwise / Pairwise / Listwise)
   - 3.2 RankNet
   - 3.3 LambdaRank и LambdaMART
   - 3.4 Triplet Loss и стратегии mining
   - 3.5 InfoNCE и контрастивное обучение
   - 3.6 BPR (Bayesian Personalized Ranking)
   - 3.7 Listwise подходы (ListNet, ListMLE)
   - 3.8 Sampling стратегии


[Хороший курс по recsys от ВШЭ](https://github.com/anamarina/RecSys_course)

# Блок 1. Метрики качества ранжирования в рекомендательных системах

## 1. Метрики качества ранжирования

### 1.1 Постановка задачи

#### Что такое ранжирование рекомендаций?

Рекомендательная система — это функция, которая для пользователя $u$ и множества объектов (items) $\mathcal{I}$ возвращает **упорядоченный** список рекомендаций. В отличие от задачи классификации, где на выходе бинарная метка или вероятность, здесь нас интересует **порядок** объектов: объект, оказавшийся выше в списке, должен быть более релевантен.

Формально: пусть $\hat{\pi}(u)$ — ранжированный список объектов, возвращаемый системой для пользователя $u$. Нас интересует **топ-K** список: первые $K$ позиций этого списка, которые пользователь видит на экране.

$$\hat{\pi}(u) = [i_1, i_2, \ldots, i_K]$$

где $i_j$ — объект на $j$-й позиции, $i_1$ считается наиболее релевантным.

#### Бинарная vs грейдированная релевантность

Релевантность объекта $i$ для пользователя $u$ может быть:

- **Бинарной**: $\text{rel}(u, i) \in \{0, 1\}$. Пользователь либо кликнул, либо нет; либо купил, либо нет. Просто, но теряет нюансы.
- **Грейдированной**: $\text{rel}(u, i) \in \{0, 1, 2, 3, \ldots\}$. Например, оценка фильма 1–5 звёзд. Грейдированная релевантность позволяет отличать «немного интересно» от «очень понравилось».


**Ключевая интуиция:** в задаче ранжирования ошибка на 1-й позиции гораздо дороже ошибки на 100-й. Метрики ранжирования учитывают это явно.

### 1.2 Precision@K и Recall@K

#### Определения и формулы

Пусть $\mathcal{R}(u)$ — множество релевантных объектов для пользователя $u$ (ground truth), а $\hat{\pi}_K(u)$ — топ-$K$ список рекомендаций.

**Precision@K** — доля релевантных объектов среди рекомендованных:

$$\text{Precision@K}(u) = \frac{|\hat{\pi}_K(u) \cap \mathcal{R}(u)|}{K}$$

**Recall@K** — доля релевантных объектов, попавших в рекомендации, от всех релевантных:

$$\text{Recall@K}(u) = \frac{|\hat{\pi}_K(u) \cap \mathcal{R}(u)|}{|\mathcal{R}(u)|}$$

#### Почему Precision@K зависит от K и числа релевантных объектов?

Precision@K нормирован на $K$, поэтому при увеличении $K$ в числитель может добавляться меньше релевантных объектов (или не добавляться вовсе), и метрика падает. **При большом K Precision@K всегда стремится к доле релевантных объектов в каталоге** — то есть к базовому уровню случайных рекомендаций.

Recall@K нормирован на $|\mathcal{R}(u)|$: при фиксированном $K$ пользователи с большим числом релевантных объектов будут иметь низкий recall — просто потому что все их интересы не уместятся в $K$ позиций.

#### Конкретный пример

> У пользователя **3 релевантных объекта** из каталога в 100 объектов. Система вернула топ-10. Среди них оказалось **2 релевантных**.

$$\text{Precision@10} = \frac{2}{10} = 0.20$$

$$\text{Recall@10} = \frac{2}{3} \approx 0.667$$

Если бы система нашла все 3 релевантных в топ-10:

$$\text{Precision@10} = \frac{3}{10} = 0.30, \quad \text{Recall@10} = \frac{3}{3} = 1.0$$

**Вывод:** Precision@K и Recall@K находятся в конфликте друг с другом (trade-off). Невозможно одновременно максимизировать оба при фиксированном $K$ — рост $K$ увеличивает Recall, но снижает Precision.

### 1.3 Average Precision и MAP

#### Мотивация

Precision@K измеряет качество ровно при одном значении $K$ и не учитывает **порядок** релевантных объектов внутри топа. Если оба релевантных объекта стоят на позициях 9 и 10, это не так хорошо, как если бы они стояли на позициях 1 и 2 — но Precision@10 в обоих случаях одинаков.

**Average Precision (AP)** устраняет эту проблему, рассматривая значения Precision при каждом попадании релевантного объекта.

#### AP как площадь под кривой P-R

Если построить кривую Precision-Recall, меняя порог $K$ от 1 до $|\mathcal{I}|$, то AP аппроксимирует площадь под этой кривой. Чем раньше система находит релевантные объекты — тем выше значения Precision в каждой точке, тем больше площадь.

#### Формула AP@K

$$\text{AP@K}(u) = \frac{1}{|\mathcal{R}(u)|} \sum_{k=1}^{K} \text{Precision@k}(u) \cdot \mathbb{1}[\text{rel}(i_k) = 1]$$

где $\mathbb{1}[\text{rel}(i_k) = 1]$ — индикатор того, что объект на позиции $k$ релевантен.

**Интерпретация:** суммируем Precision@k только в тех позициях, где встретили релевантный объект, а затем нормируем на число релевантных объектов.

#### MAP: усреднение по пользователям

**Mean Average Precision (MAP)** — это макроусреднение AP по всем пользователям:

$$\text{MAP@K} = \frac{1}{|U|} \sum_{u \in U} \text{AP@K}(u)$$

**Зачем макроусреднение?** При микроусреднении (усреднение по всем объектам) пользователи с большим числом взаимодействий доминируют в метрике. Макроусреднение гарантирует, что каждый пользователь вносит одинаковый вклад. Это особенно важно при «длинном хвосте» активности: не позволять 5% активных пользователей определять качество системы для всех.

#### Пример вычисления вручную

> Пользователь имеет 3 релевантных объекта. Топ-5 рекомендаций системы:

| Позиция $k$ | Объект | Релевантен? | Precision@k | Слагаемое |
|---|---|---|---|---|
| 1 | A | Да | 1/1 = 1.000 | 1.000 |
| 2 | B | Нет | 1/2 = 0.500 | — |
| 3 | C | Да | 2/3 = 0.667 | 0.667 |
| 4 | D | Нет | 2/4 = 0.500 | — |
| 5 | E | Да | 3/5 = 0.600 | 0.600 |

$$\text{AP@5} = \frac{1}{3}(1.000 + 0.667 + 0.600) = \frac{2.267}{3} \approx 0.756$$

Если бы система поставила все три релевантных объекта на позиции 3, 4, 5 (а не 1, 3, 5):

$$\text{AP@5} = \frac{1}{3}\left(\frac{1}{3} + \frac{2}{4} + \frac{3}{5}\right) = \frac{1}{3}(0.333 + 0.500 + 0.600) \approx 0.478$$

Тот же набор объектов в топ-5 — но AP значительно ниже, потому что релевантные объекты встречаются позже.

### 1.4 NDCG (Normalized Discounted Cumulative Gain)

#### Мотивация

MAP работает с бинарной релевантностью. Но если пользователь оценил фильм на 5 звёзд, это не то же самое, что «поставил лайк». NDCG обобщает идею AP на **грейдированную релевантность** и явно **дисконтирует объекты по их позиции** в списке.

**Ключевая идея:** вклад объекта на позиции $k$ уменьшается логарифмически с ростом $k$. Объект на 1-й позиции приносит вдвое больше «пользы», чем на 3-й.

#### DCG (Discounted Cumulative Gain)

$$\text{DCG@K} = \sum_{k=1}^{K} \frac{2^{\text{rel}(i_k)} - 1}{\log_2(k + 1)}$$

где $\text{rel}(i_k)$ — грейдированная релевантность объекта на позиции $k$.

Альтернативная (упрощённая) формула, часто встречаемая в литературе:

$$\text{DCG@K} = \sum_{k=1}^{K} \frac{\text{rel}(i_k)}{\log_2(k + 1)}$$

Первая формула усиливает вклад высоко релевантных объектов через экспоненту $2^{\text{rel}}$.

#### IDCG (Ideal DCG)

IDCG — это DCG для **идеального** ранжирования: объекты упорядочены по убыванию релевантности.

$$\text{IDCG@K} = \sum_{k=1}^{K} \frac{2^{\text{rel}^*(i_k)} - 1}{\log_2(k + 1)}$$

где $\text{rel}^*(i_k)$ — релевантность $k$-го объекта в идеальном порядке.

#### NDCG (Normalized DCG)

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

NDCG $\in [0, 1]$: значение 1 означает идеальный порядок, 0 — полное отсутствие релевантных объектов в топе.

#### Логарифмический дискаунт: интерпретация

Знаменатель $\log_2(k+1)$ имеет содержательную интерпретацию: он моделирует **убывающее внимание пользователя**. Исследования в области информационного поиска (eye-tracking, click-through rate) показывают, что пользователи просматривают позиции нелинейно: первую позицию видят почти все, пятую — уже значительно меньше людей.

```
Позиция k:     1      2      3      4      5      10
log2(k+1):   1.00   1.58   2.00   2.32   2.58   3.46
1/log2(k+1): 1.00   0.63   0.50   0.43   0.39   0.29
```

Дискаунт убывает быстро в начале списка и замедляется по мере роста позиции — что соответствует реальному поведению пользователей.

#### Пример с грейдированными оценками

> Каталог фильмов. У пользователя оценки: A=3, B=0, C=2, D=1, E=0. Система вернула: [A, B, C, D, E].

| Позиция $k$ | Объект | $$\text{rel}$$ | $$2^{\text{rel}}-1$$ | $$\log_2(k+1)$$ | Вклад в DCG |
|---|---|---|---|---|---|
| 1 | A | 3 | 7 | 1.000 | 7.000 |
| 2 | B | 0 | 0 | 1.585 | 0.000 |
| 3 | C | 2 | 3 | 2.000 | 1.500 |
| 4 | D | 1 | 1 | 2.322 | 0.431 |
| 5 | E | 0 | 0 | 2.585 | 0.000 |

$$\text{DCG@5} = 7.000 + 0 + 1.500 + 0.431 + 0 = 8.931$$

Идеальный порядок: [A(3), C(2), D(1), B(0), E(0)]

| Позиция $k$ | $$\text{rel}^*$$ | $$2^{\text{rel}^*}-1$$ | $$\log_2(k+1)$$ | Вклад в IDCG |
|---|---|---|---|---|
| 1 | 3 | 7 | 1.000 | 7.000 |
| 2 | 2 | 3 | 1.585 | 1.893 |
| 3 | 1 | 1 | 2.000 | 0.500 |
| 4 | 0 | 0 | 2.322 | 0.000 |
| 5 | 0 | 0 | 2.585 | 0.000 |

$$\text{IDCG@5} = 7.000 + 1.893 + 0.500 + 0 + 0 = 9.393$$

$$\text{NDCG@5} = \frac{8.931}{9.393} \approx 0.951$$

Высокое значение: система почти идеально ранжировала объекты (только C и D поменялись местами с нерелевантным B).

### 1.5 MRR (Mean Reciprocal Rank)

#### Формула

Для пользователя $u$ пусть $r_u$ — ранг (позиция) **первого** релевантного объекта в списке рекомендаций.

$$\text{RR}(u) = \frac{1}{r_u}$$

$$\text{MRR} = \frac{1}{|U|} \sum_{u \in U} \frac{1}{r_u}$$

Если ни одного релевантного объекта в списке нет, полагают $\text{RR}(u) = 0$.

#### Когда MRR уместна?

MRR логична в задачах, где пользователь ищет **один конкретный объект** и удовлетворён, как только его нашёл:

- Поисковые системы: «ближайшая аптека», «столица Бразилии»
- Голосовые ассистенты: один ответ на вопрос
- Рекомендации следующего трека в плейлисте

Примеры значений MRR:

```
Первый релевантный на позиции 1:  RR = 1.0
Первый релевантный на позиции 2:  RR = 0.5
Первый релевантный на позиции 5:  RR = 0.2
Первый релевантный на позиции 10: RR = 0.1
```

#### Ограничения MRR

- **Учитывается только первый релевантный объект.** Если у пользователя 20 релевантных объектов, MRR игнорирует 19 из них. В контексте рекомендаций фильмов или товаров это неприемлемо.
- **Нечувствительна к различиям после первой позиции.** Разница между «первый релевантный на позиции 3» и «первый на позиции 4» одинаково важна для MRR, но разница между «на позиции 1» и «на позиции 2» — вдвое больше.
- Подходит только для сценариев с **единственной целью** поиска.

### 1.6 Hit Rate (HR@K)

#### Определение и формула

Hit Rate@K (иногда называемый Recall@K при одном целевом объекте) отвечает на вопрос: **попал ли хотя бы один релевантный объект в топ-K?**

$$\text{HR@K}(u) = \mathbb{1}[|\hat{\pi}_K(u) \cap \mathcal{R}(u)| \geq 1]$$

$$\text{HR@K} = \frac{1}{|U|} \sum_{u \in U} \text{HR@K}(u)$$

#### Отличие от Recall@K

В контексте рекомендательных систем часто используется **протокол leave-one-out**: из истории пользователя убирается один объект (последний в хронологии), и задача системы — угадать его среди $K$ рекомендаций. В этом случае $|\mathcal{R}(u)| = 1$, и:

$$\text{Recall@K}(u) = \text{HR@K}(u)$$

Однако при $|\mathcal{R}(u)| > 1$ метрики расходятся:

- **Recall@K** показывает долю всех релевантных объектов, попавших в топ
- **HR@K** бинарен: засчитывается факт хотя бы одного попадания

HR@K менее строга и проще для интерпретации в сценариях «следующей покупки» или «следующего клика».

### 1.7 Diversity и Coverage

До сих пор мы рассматривали метрики, измеряющие **точность** (accuracy) рекомендаций. Но у рекомендательной системы есть и другие измерения качества.

#### Intra-List Diversity

Если система рекомендует 10 фильмов — все боевики с одним актёром — технически она может иметь высокий NDCG. Но пользователь хочет разнообразия.

**Intra-List Diversity (ILD)** — средняя попарная непохожесть объектов в списке рекомендаций:

$$\text{ILD}(L_u) = \frac{2}{|L_u|(|L_u|-1)} \sum_{\substack{i, j \in L_u \\ i \neq j}} d(i, j)$$

где $d(i, j)$ — расстояние (диссimilarity) между объектами $i$ и $j$ (например, $1 - \text{cosine\_similarity}(i, j)$ в пространстве признаков).

Чем выше ILD, тем разнообразнее список.

#### Catalog Coverage

**Catalog Coverage** — доля уникальных объектов каталога, которые система хотя бы кому-то рекомендовала:

$$\text{CovCatalog} = \frac{\left|\bigcup_{u \in U} \hat{\pi}_K(u)\right|}{|\mathcal{I}|}$$

Низкое покрытие каталога означает, что система рекомендует одни и те же «хиты» всем подряд, игнорируя длинный хвост (long tail). Это невыгодно и для пользователей с нишевыми интересами, и для платформы (меньше монетизируется контент).

#### User Coverage

**User Coverage** — доля пользователей, для которых система смогла сгенерировать хотя бы $K$ рекомендаций:

$$\text{CovUsers} = \frac{|\{u \in U : |\hat{\pi}_K(u)| \geq K\}|}{|U|}$$

Актуально для систем с холодным стартом: новые пользователи или объекты могут не иметь достаточно данных.

#### Accuracy-Diversity Dilemma

Существует фундаментальное противоречие между точностью и разнообразием:

> **Чем лучше система «знает» предпочтения пользователя, тем более однородные рекомендации она выдаёт.**

Это объясняется просто: если система уверена, что пользователь любит триллеры, она будет рекомендовать именно триллеры — что максимизирует accuracy, но снижает ILD. На практике используют **многокритериальную оптимизацию** или **диверсификацию post-hoc** — например, алгоритм Maximal Marginal Relevance (MMR):

$$\text{MMR}(i) = \lambda \cdot \text{relevance}(u, i) - (1 - \lambda) \cdot \max_{j \in S} \text{sim}(i, j)$$

где $S$ — уже выбранные объекты, $\lambda \in [0, 1]$ — параметр баланса точность/разнообразие.

### 1.8 Beyond-Accuracy Метрики

Рекомендательные системы оцениваются не только по точности попадания в предпочтения. Ряд метрик фиксирует более тонкие аспекты пользовательского опыта.

#### Novelty (Новизна / Неожиданность)

**Novelty** измеряет, насколько рекомендованные объекты **неизвестны** или **непопулярны** для пользователя. Часто формализуется через обратную популярность:

$$\text{Novelty}(u) = \frac{1}{K} \sum_{i \in \hat{\pi}_K(u)} -\log_2 p(i)$$

где $p(i)$ — вероятность того, что случайный пользователь взаимодействовал с объектом $i$ (мера популярности). Чем менее популярен объект, тем выше его вклад в novelty.

**Проблема:** высокая novelty ≠ высокое качество. Система может рекомендовать неизвестные объекты, которые пользователю не понравятся.

#### Serendipity (Серендипность)

**Serendipity** — «приятная неожиданность»: объект одновременно **неожиданный** (отличается от предыдущих взаимодействий пользователя) и **релевантный** (пользователю понравился).

$$\text{Serendipity}(u) = \frac{1}{K} \sum_{i \in \hat{\pi}_K(u)} \mathbb{1}[\text{unexpected}(u, i)] \cdot \mathbb{1}[\text{relevant}(u, i)]$$

Формально «неожиданность» можно определить как малую похожесть объекта на профиль пользователя или на предсказания наивной baseline-системы.

**Пример:** пользователь смотрит только боевики, но ему неожиданно понравился документальный фильм о природе — это серендипная рекомендация.

#### Fairness (Справедливость)

По мере распространения рекомендательных систем всё большую роль играет **Fairness** — обеспечение того, чтобы система не дискриминировала пользователей или поставщиков контента по чувствительным признакам.

Различают два уровня:

- **User-side fairness**: все группы пользователей (по полу, возрасту, географии) получают одинаково качественные рекомендации.
- **Item-side fairness (provider fairness)**: контент разных производителей (малые vs крупные студии, меньшинства vs мейнстрим) получает справедливую экспозицию.

Типичная формализация — через Exposure Fairness:

$$\text{ExposureDisparity} = \left| \mathbb{E}[\text{Exposure}(\text{group}_A)] - \mathbb{E}[\text{Exposure}(\text{group}_B)] \right|$$

Тема fairness в RecSys активно исследуется и выходит за рамки технических метрик — требует нормативных решений о том, что считать «справедливым».

### 1.9 Как выбирать метрику

Выбор метрики — это не технический, а **содержательный** вопрос: что именно важно для вашего продукта?

#### Таблица: метрика → когда применять → ограничения

| Метрика | Когда применять | Ограничения |
|---|---|---|
| **Precision@K** | Когда важно, что именно в топе; небольшие $K$ | Зависит от $K$; игнорирует порядок внутри топа |
| **Recall@K** | Когда важно покрытие всех интересов пользователя | Зависит от числа релевантных объектов |
| **MAP** | Когда нужно учесть порядок и несколько релевантных объектов | Только бинарная релевантность; чувствителен к хвосту |
| **NDCG@K** | Грейдированная релевантность; позиция критична | Требует размеченных оценок; чувствителен к выбору $K$ |
| **MRR** | Один целевой объект; поиск, Q&A | Игнорирует всё, кроме первого попадания |
| **HR@K** | Leave-one-out протокол; binary hit/miss достаточен | Нечувствителен к порядку и числу попаданий |
| **ILD** | Диверсификация списка; борьба с filter bubble | Не связана с релевантностью |
| **Catalog Coverage** | Монетизация длинного хвоста | Не отражает качество рекомендаций |
| **Novelty** | Исследование новых жанров; борьба с popularity bias | Высокая novelty ≠ релевантность |
| **Serendipity** | UX; удержание пользователей через приятные сюрпризы | Сложно формализовать; нет единого стандарта |

#### Офлайн vs Онлайн (A/B): критически важное различие

**Офлайн-метрики** вычисляются на исторических данных. Они быстрые и воспроизводимые, но страдают от системных проблем:

- **Survivorship bias**: мы знаем только о тех взаимодействиях, которые произошли. Если система никогда не рекомендовала объект $X$, мы не знаем, понравился бы он пользователю.
- **Exposure bias**: пользователи могут взаимодействовать только с тем, что им показали. Офлайн-оценка не учитывает это.
- **Метрики не коррелируют с бизнес-целями** напрямую: рост NDCG на 2% не гарантирует рост кликов или выручки.

**A/B-тест** — единственный способ измерить реальный эффект на поведение пользователей. Но он требует трафика, времени и инфраструктуры.

**Практический вывод:**

> Используйте офлайн-метрики для быстрой итерации и отбора кандидатов. Финальное решение о деплое — только через A/B-тест с бизнес-метриками (CTR, конверсия, время на платформе, retention).

### Итог

Метрики качества ранжирования в рекомендательных системах образуют иерархию:

1. **Базовые метрики точности** (Precision@K, Recall@K) — просты и интерпретируемы, но неполны.
2. **Метрики с учётом порядка** (MAP, NDCG, MRR) — учитывают позицию объекта в списке; выбор зависит от типа релевантности и числа целевых объектов.
3. **Метрики за пределами точности** (Diversity, Coverage, Novelty, Serendipity, Fairness) — отражают долгосрочное качество системы и пользовательский опыт.

Ни одна метрика не является универсальной. Зрелая рекомендательная система оценивается **набором** метрик, балансирующих accuracy с diversity, coverage и fairness — и всегда в конечном счёте верифицируется через онлайн-эксперименты.

---

# Блок 2. Валидация в рекомендательных системах

**Цель блока:** разобрать, почему стандартные практики валидации из классического ML неприменимы к рекомендательным системам напрямую, изучить корректные протоколы оценки и научиться избегать типичных ловушек.


## 2.1 Почему стандартный random split не работает

### Взаимодействия — это временные ряды

Данные о взаимодействиях пользователей с объектами (просмотры, клики, покупки, оценки) имеют принципиально иную структуру по сравнению с типичными задачами классификации или регрессии. Каждое взаимодействие имеет **временну́ю метку** — оно произошло в определённый момент и несёт в себе причинно-следственную зависимость: пользователь взаимодействовал с объектом, потому что в момент взаимодействия его предпочтения, информационный контекст и доступный ассортимент были именно такими.

Формально, набор взаимодействий пользователя $u$ можно представить как упорядоченный во времени список:

$$
\mathcal{H}_u = \{(i_1, t_1),\, (i_2, t_2),\, \ldots,\, (i_n, t_n)\}, \quad t_1 < t_2 < \cdots < t_n
$$

где $i_k$ — объект (item), $t_k$ — временна́я метка взаимодействия. Задача рекомендательной системы — предсказать будущие взаимодействия $i_{n+1}, i_{n+2}, \ldots$ на основе истории $\mathcal{H}_u$.

### Data leakage при random split

Стандартный `train_test_split(shuffle=True)` случайным образом перемешивает все взаимодействия и распределяет их по множествам. При этом в обучающую выборку может попасть взаимодействие, произошедшее **позже**, чем тестовое. Модель обучается «знать будущее» и предсказывает «прошлое» — это классический случай **data leakage**.

**Пример.** Пусть пользователь посмотрел:
- $t_1$: «Дюна: Часть 1»
- $t_2$: «Дюна: Часть 2»
- $t_3$: «Властелин колец»

При random split $(t_3)$ может оказаться в train, а $(t_2)$ — в test. Модель обучается на «будущем» поведении пользователя (он уже посмотрел «Властелин колец») и предсказывает «прошлое» («Дюна: Часть 2»). Задача упрощается, а метрики искусственно завышаются.

### Последствия в практике

Эмпирически показано ([Campos et al., RecSys 2020](https://arxiv.org/pdf/2007.13237)), что **ранжирование моделей по качеству меняется** в зависимости от стратегии сплита. Модель, показывающая SOTA на random split, может проигрывать базовой линии при temporal split. Это означает, что результаты сотен академических статей, использовавших random split, потенциально некорректны и несопоставимы между собой.

| Стратегия сплита | Утечка данных | Реалистичность | Применимость |
|---|---|---|---|
| Random split | Высокая | Низкая | ❌ Не рекомендуется |
| Global temporal split | Нет | Высокая | ✅ Промышленный стандарт |
| Per-user temporal split | Низкая* | Средняя | ✅ При осторожном применении |
| Leave-one-out (случайный) | Высокая | Низкая | ❌ Устаревший |
| Leave-last-out (временной) | Нет | Высокая | ✅ Академический стандарт |

*При несинхронных пользовательских порогах возникает частичная утечка трендов.

> ⚠️ **Антипаттерн: `sklearn.train_test_split` без временно́й сортировки**
>
> ```python
> # НЕПРАВИЛЬНО — перемешивает временну́ю структуру
> from sklearn.model_selection import train_test_split
> train, test = train_test_split(interactions_df, test_size=0.2, random_state=42)
>
> # ПРАВИЛЬНО — сохраняем временну́ю последовательность
> interactions_df = interactions_df.sort_values('timestamp')
> threshold = interactions_df['timestamp'].quantile(0.8)
> train = interactions_df[interactions_df['timestamp'] <= threshold]
> test  = interactions_df[interactions_df['timestamp'] >  threshold]
> ```

## 2.2 Temporal Split

### Global temporal split

**Global temporal split** — наиболее строгий и реалистичный протокол. Выбирается глобальный временно́й порог $T^*$, который делит всю историю взаимодействий на train и test:

$$
\mathcal{D}_{\text{train}} = \{(u, i, t) \mid t \leq T^*\}, \quad \mathcal{D}_{\text{test}} = \{(u, i, t) \mid t > T^*\}
$$

Обычно также вводится **validation window** — второй порог $T^{**} < T^*$:

$$
\mathcal{D}_{\text{train}} = \{t \leq T^{**}\}, \quad \mathcal{D}_{\text{val}} = \{T^{**} < t \leq T^*\}, \quad \mathcal{D}_{\text{test}} = \{t > T^*\}
$$

**Как выбирать $T^*$?** Типичные соотношения в промышленности: 70–80% временного диапазона на train, 10–15% на val, 10–15% на test. Однако важнее не доля, а **наличие достаточного количества взаимодействий** в каждом разделе для статистически значимых оценок.

**Проблема холодных пользователей в тесте.** Если пользователь зарегистрировался после $T^*$, в train нет его истории. Это «холодный» пользователь в тесте. Необходимо либо **исключать** таких пользователей из тестовой оценки (оцениваем только known-user performance), либо **оценивать отдельно** (cold-start scenario).

### Per-user temporal split

**Per-user temporal split** устанавливает индивидуальный порог для каждого пользователя. Наиболее распространённый вариант — **Leave-last-$N$-out**:

$$
\mathcal{D}_{\text{test}}^{(u)} = \{(u, i_k, t_k) \mid k > |\mathcal{H}_u| - N\}
$$

То есть для каждого пользователя последние $N$ взаимодействий (самые поздние по времени) отправляются в тест, всё остальное — в train.

При $N = 1$ получаем классический **Leave-last-one-out** протокол:

$$
\mathcal{D}_{\text{test}}^{(u)} = \{(u,\, i_{|\mathcal{H}_u|},\, t_{|\mathcal{H}_u|})\}
$$

**Проблема синхронизации.** При per-user split пользователь А мог совершить своё «последнее» взаимодействие в январе, а пользователь Б — в декабре. Взаимодействия января пользователя Б попадают в train и потенциально описывают тренды, которые модель «переносит» на тест пользователя А. Это **частичная утечка временны́х трендов** ([Campos et al., 2020](https://arxiv.org/pdf/2007.13237)).

### Как выбрать окно: user activity distribution

При выборе параметров сплита необходимо анализировать **распределение активности пользователей**. Типичная ситуация в реальных датасетах — Zipf-подобное распределение: малая доля пользователей генерирует большую часть взаимодействий.

Практический алгоритм:
1. Постройте гистограмму $|\mathcal{H}_u|$ для всех пользователей.
2. Найдите медианное количество взаимодействий на пользователя.
3. Убедитесь, что после сплита **каждый пользователь** в тесте имеет хотя бы $K_{\min}$ взаимодействий в train (обычно $K_{\min} = 5$).
4. Пользователей с $|\mathcal{H}_u| \leq K_{\min} + N$ следует исключить из оценки или рассматривать отдельно как cold-start.

```python
# Фильтрация пользователей с недостаточной историей
MIN_TRAIN_INTERACTIONS = 5
N_TEST = 1

valid_users = (
    interactions_df
    .groupby('user_id')['item_id']
    .count()
    .loc[lambda x: x >= MIN_TRAIN_INTERACTIONS + N_TEST]
    .index
)
interactions_df = interactions_df[interactions_df['user_id'].isin(valid_users)]
```

> ⚠️ **Антипаттерн: оценка на всех пользователях без фильтрации**
>
> Если оставлять пользователей с 1–2 взаимодействиями, тестовый объект может стать **единственным известным** пользователю. Модель вырождается в задачу «угадай единственное взаимодействие пользователя», что нереалистично и даёт нестабильные оценки.

## 2.3 Протоколы валидации

### Leave-One-Out (LOO)

**LOO** — исторически первый и наиболее популярный в академической литературе протокол. Для каждого пользователя одно взаимодействие (как правило, последнее) используется как тестовое:

$$
\forall u:\quad |\mathcal{D}_{\text{test}}^{(u)}| = 1
$$

**Преимущества:**
- Воспроизводимость: чёткий детерминированный алгоритм формирования тестовых множеств.
- Максимальное использование данных: почти весь датасет участвует в обучении.
- Широко принят в статьях по RecSys, что облегчает сравнение с baseline-ами.

**Недостатки:**
- **Нереалистичность:** в реальной системе мы оцениваем поведение пользователя на горизонте дней/недель, а не угадываем один конкретный следующий клик.
- **Высокая дисперсия:** одно тестовое взаимодействие — ненадёжная оценка предпочтений пользователя. Случайный шум (пользователь кликнул нечаянно) приводит к нестабильным метрикам.
- **Завышенные метрики:** протокол слишком прост — нередко достаточно запомнить «типичный следующий объект» для пользовательского кластера.

> ⚠️ **Антипаттерн: LOO без временно́й сортировки**
>
> В некоторых ранних статьях «leave-one-out» означало выбор **случайного** взаимодействия, а не последнего по времени. Это нарушает временну́ю структуру и вводит data leakage. Всегда уточняйте: «last» или «random» LOO.

### Holdout с временным сплитом: промышленный стандарт

В индустрии используется **temporal holdout**:
1. Данные разбиваются по времени на train / val / test.
2. Гиперпараметры подбираются только по val.
3. Финальная оценка производится на test **один раз**.

Это соответствует реальному сценарию деплоя: модель обучается на прошлых данных и предсказывает будущее.

### K-fold для RecSys: почему сложнее, чем в классическом ML

В классическом ML K-fold cross-validation делит датасет на $K$ равных частей и поочерёдно использует каждую как test. В RecSys это **неприменимо напрямую** по нескольким причинам:

1. **Нарушение временно́й структуры.** Если fold 3 содержит более ранние данные, чем fold 1, то при оценке fold 3 модель «знает будущее».

2. **Связность пользователей и объектов.** В fold для теста могут оказаться пользователи, отсутствующие в train — это искусственный cold-start.

3. **Вычислительная стоимость.** Рекомендательные модели дороги в обучении; $K$-кратное переобучение нецелесообразно.

**Корректная адаптация — temporal K-fold:** $K$ последовательных временны́х окон, каждое из которых обучает модель на всём предшествующем периоде:

$$
\text{fold}_k: \quad \mathcal{D}_{\text{train}} = \{t \leq T_k\},\quad \mathcal{D}_{\text{test}} = \{T_k < t \leq T_{k+1}\}
$$

Это называется **expanding window** (расширяющееся окно) и даёт несмещённую оценку устойчивости модели во времени.

## 2.4 Проблема популярных объектов (Popularity Bias)

### Феномен «длинного хвоста»

Распределение популярности объектов в рекомендательных системах подчиняется **законам степенного распределения** (Zipf, Pareto): небольшое число объектов собирает подавляющее большинство взаимодействий, а огромный «длинный хвост» из нишевых объектов остаётся практически неизвестным.

Формально, если $c_i$ — количество взаимодействий объекта $i$, а объекты отсортированы по убыванию популярности, то:

$$
c_i \propto i^{-\alpha}, \quad \alpha \approx 1 \text{ (закон Ципфа)}
$$

**Последствия для оценки:**

- Стандартные метрики (Precision@K, Recall@K, NDCG@K) считаются **в среднем по всем тестовым пользователям**. Поскольку большинство тестовых объектов — это популярные объекты (у них больше взаимодействий), метрики де-факто **измеряют качество на head items**.
- Длиннохвостые объекты практически не влияют на агрегированные метрики. Модель, рекомендующая только популярные объекты (TopPop baseline), может показывать конкурентные результаты по стандартным метрикам.

### Stratified evaluation по популярности

Правильный подход — **оценивать модель в разбивке по группам популярности**:

| Группа | Определение | Типичная доля объектов | Типичная доля взаимодействий |
|---|---|---|---|
| Head | Топ-20% по популярности | 20% | ~80% |
| Mid-tail | 20–50% по популярности | 30% | ~15% |
| Long-tail | Нижние 50% | 50% | ~5% |

```python
# Пример: стратифицированная оценка
def stratified_ndcg(predictions, test_data, item_popularity):
    groups = pd.qcut(item_popularity, q=[0, 0.5, 0.8, 1.0],
                     labels=['long_tail', 'mid_tail', 'head'])
    results = {}
    for group_name in ['head', 'mid_tail', 'long_tail']:
        group_items = item_popularity[groups == group_name].index
        group_test = test_data[test_data['item_id'].isin(group_items)]
        results[group_name] = compute_ndcg(predictions, group_test)
    return results
```

### Popularity-debiased метрики

Один из подходов — **взвешивание тестовых объектов обратно пропорционально их популярности**:

$$
\text{NDCG}_{\text{debias}} = \frac{1}{|U|} \sum_{u} \sum_{i \in \mathcal{T}_u} \frac{1}{c_i} \cdot \frac{\text{rel}(u,i)}{\log_2(\text{rank}(u,i) + 1)}
$$

Другой подход — **равномерная выборка тестовых объектов** из разных корзин популярности, чтобы все группы были одинаково представлены в оценке.

> ⚠️ **Антипаттерн: сравнение моделей только по средним метрикам**
>
> Модель A: NDCG@10 = 0.142 (высокое качество на head, плохо на tail)
> Модель B: NDCG@10 = 0.138 (равномерно по всем группам)
>
> По стандартной метрике «лучше» модель A. Но если бизнес-задача — монетизация нишевого контента, нужна модель B. Всегда приводите стратифицированные результаты.

## 2.5 Exposure Bias и позиционный bias

### Смещение экспозиции: формализация через MNAR

Данные о взаимодействиях в рекомендательных системах никогда не являются случайной выборкой из пространства всех возможных предпочтений. Пользователь видит только те объекты, которые ему **показала** система — и взаимодействует только с ними. Взаимодействие с объектом $i$ пользователем $u$ наблюдается только при условии, что объект был экспонирован.

Введём индикатор экспозиции: $O_{ui} = 1$, если пользователь $u$ видел объект $i$, иначе $O_{ui} = 0$. Тогда:

$$
P(O_{ui} = 1) \neq \text{const}
$$

Это и есть **Missing Not At Random (MNAR)** — отсутствие взаимодействия не означает, что пользователю не понравился объект; возможно, он его просто не видел. Формально, пусть $R_{ui} \in \{0, 1\}$ — истинная релевантность (понравится/не понравится), а $Y_{ui}$ — наблюдаемое взаимодействие. Тогда:

$$
Y_{ui} = R_{ui} \cdot O_{ui}
$$

При MNAR: $P(O_{ui} = 1 \mid R_{ui} = 1) \neq P(O_{ui} = 1 \mid R_{ui} = 0)$ — вероятность наблюдения взаимодействия зависит от самой релевантности (и от популярности объекта).

**Ключевое следствие:** пространство «неизвестных» взаимодействий $\{(u,i) : Y_{ui} = 0\}$ содержит два принципиально разных типа:
1. Объект **не понравился** бы пользователю (истинный негатив).
2. Объект **понравился** бы, но не был показан (ненаблюдаемый позитив).

> ⚠️ **Антипаттерн: unobserved = negative**
>
> Многие реализации неявно трактуют все ненаблюдаемые взаимодействия как отрицательные при обучении. Это нарушает MNAR-предположение и приводит к штрафованию модели за рекомендацию объектов, которые пользователь не видел, но мог бы полюбить.

### IPS-коррекция

**Inverse Propensity Scoring (IPS)** — метод из теории причинного вывода, адаптированный для RecSys. Идея: взвесить каждое наблюдённое взаимодействие обратно пропорционально **вероятности его наблюдения (propensity)**:

$$
\hat{L}_{\text{IPS}}(\hat{R}) = \frac{1}{|\mathcal{D}_{\text{obs}}|} \sum_{(u,i) \in \mathcal{D}_{\text{obs}}} \frac{\ell(Y_{ui},\, \hat{R}_{ui})}{p_{ui}}
$$

где $p_{ui} = P(O_{ui} = 1)$ — propensity score (вероятность экспозиции), $\ell$ — функция потерь.

**Свойство несмещённости IPS-оценщика:**

$$
\mathbb{E}\left[\hat{L}_{\text{IPS}}(\hat{R})\right] = L_{\text{ideal}}(\hat{R}) = \frac{1}{|U||I|} \sum_{u,i} \ell(R_{ui},\, \hat{R}_{ui})
$$

**Практические ограничения IPS:**
- Требуется знать или оценивать $p_{ui}$ — на практике их оценивают через логарифм частоты взаимодействий: $p_{i} \propto c_i^\beta$, $\beta \in (0,1)$.
- При малых $p_{ui}$ дисперсия IPS взрывается. Используется **клиппирование**: $\tilde{p}_{ui} = \max(p_{ui}, p_{\min})$.
- **Self-Normalized IPS (SNIPS)** снижает дисперсию ценой малого смещения:

$$
\hat{L}_{\text{SNIPS}}(\hat{R}) = \frac{\sum_{(u,i) \in \mathcal{D}_{\text{obs}}} \frac{\ell(Y_{ui}, \hat{R}_{ui})}{p_{ui}}}{\sum_{(u,i) \in \mathcal{D}_{\text{obs}}} \frac{1}{p_{ui}}}
$$

### Позиционный bias

Помимо exposure bias, существует **позиционный (position) bias**: пользователь с большей вероятностью кликает на объект, показанный на первых позициях, независимо от его релевантности. Это приводит к **самосбывающемуся пророчеству**: система учится показывать наверх то, что исторически получало клики, что само по себе объясняется позицией.

Корректный учёт: **позиционное взвешивание** при расчёте метрик или использование **counterfactual logging** с рандомизацией позиций.

## 2.6 Evaluation на имплицитных данных

### Implicit vs Explicit Feedback

| Тип фидбека | Примеры | Интерпретация |
|---|---|---|
| **Explicit** | Оценки (1–5 звёзд), лайки/дизлайки | Прямое выражение предпочтений |
| **Implicit** | Клики, просмотры, покупки, время на странице | Косвенный сигнал, неоднозначный |

Implicit feedback преобладает в современных промышленных системах: пользователи редко ставят явные оценки, но генерируют миллиарды неявных сигналов. Однако интерпретация затруднена:

- **Клик ≠ понравилось:** кликбейт-заголовки провоцируют клики на нерелевантный контент.
- **Отсутствие клика ≠ не понравилось:** объект мог не быть показан (MNAR).
- **Высокое время просмотра**: может означать как высокую вовлечённость, так и сложность контента или технические проблемы.

### Что значит «релевантный» при только лайках/кликах

В контексте оценки рекомендательной системы принято считать **релевантным** тот объект, с которым пользователь совершил целевое взаимодействие в тестовом периоде. Это **proxy**: мы не знаем истинных предпочтений, но аппроксимируем их через наблюдённое поведение.

**Трактовка ненаблюдаемых взаимодействий при оценке:**
- *Стратегия «all items»:* все неизвестные объекты считаются нерелевантными. Вычислительно дорого ($O(|I|)$ на пользователя).
- *Стратегия сэмплирования негативов:* для оценки ранжируются только тестовый объект и $K$ случайно выбранных нерелевантных.

### 100-item evaluation: протокол и его критика

Стандартный академический протокол **100-item evaluation** ([He et al., 2017](https://arxiv.org/abs/1708.05031)):
1. Для каждого тестового взаимодействия $(u, i^+)$ случайно сэмплируются 99 негативных объектов $\{i_1^-, \ldots, i_{99}^-\}$.
2. Метрики (HR@10, NDCG@10) считаются по ранжированию в этом пуле из 100 объектов.

**Математическая критика (Krichene & Rendle, KDD 2020 / CACM 2022):**

Пусть $M$ — истинная метрика (по всему каталогу), $\hat{M}$ — сэмплированная (по 100 объектам). Авторы доказывают, что $\hat{M}$ **несостоятельна** относительно $M$:

$$
\exists\, A, B:\quad M(A) > M(B) \quad \text{но} \quad \mathbb{E}[\hat{M}(A)] < \mathbb{E}[\hat{M}(B)]
$$

То есть, даже в ожидании (при неограниченном количестве сэмплов), ранжирование моделей по сэмплированным метрикам может **инвертировать** ранжирование по точным метрикам. Чем меньше размер пула сэмплов, тем сильнее все метрики коллапсируют к AUC (позиционные веса теряют смысл).

$$
\hat{\text{NDCG}}_k^{(S)} \xrightarrow{|S| \to 1} \text{AUC}
$$

**Практический вывод:** избегайте 100-item evaluation для сравнения моделей. Используйте **full ranking** (ранжирование по всему каталогу) или применяйте поправку Krichene–Rendle.

> ⚠️ **Антипаттерн: HR@10 из 100 кандидатов вместо HR@10 из полного каталога**
>
> Эти метрики измеряют разные вещи. HR@10/100 — вероятность того, что правильный объект окажется в топ-10 случайного пула. HR@10/|I| — вероятность того, что модель в целом помещает правильный объект в топ-10. Первая метрика всегда выше и не коррелирует со второй одинаково для всех моделей.

## 2.7 Офлайн vs Онлайн метрики

### Discrepancy: почему офлайн NDCG не коррелирует с CTR

Офлайн-метрики (NDCG, Recall, Precision) оцениваются на **статичном снимке данных**. Онлайн-метрики (CTR, Dwell Time, Revenue, Retention) измеряются в **динамической системе** с реальными пользователями. Разрыв между ними обусловлен следующими причинами:

1. **MNAR bias** (см. 4.5): тестовые данные смещены в сторону популярных объектов. Модель, хорошо предсказывающая смещённые тестовые данные, не обязательно лучше рекомендует нехоженые объекты.

2. **Распределение данных смещается со временем:** офлайн-данные могут быть устаревшими; модель, обученная на прошлом, сталкивается с изменившимися предпочтениями пользователей.

3. **Позиционный bias не учтён:** NDCG предполагает, что пользователь видит весь список, но в реальности CTR сильно зависит от позиции.

4. **Многоцелевая природа реальных систем:** пользователь может кликнуть на объект не потому что он релевантен, а потому что заголовок кликабельный — CTR ≠ удовлетворённость.

Исследование ([Jannach et al., 2020](https://dynamicdecisions.github.io/assets/pdfs/12.pdf)) показывает, что NDCG и RMSE коррелируют с реальными пользовательскими рейтингами, но эта корреляция нарушается, когда модель меняет **распределение контента** (при деплое появляются новые паттерны взаимодействий, которых нет в офлайн-данных).

Ключевой результат работы ([Saito et al., 2022](https://arxiv.org/html/2307.15053v3)): **ненормализованный DCG** (без нормировки на идеальный список) коррелирует с онлайн-метриками значительно лучше, чем стандартный **nDCG**. Нормализация рендерит метрику **несостоятельной**: она может инвертировать порядок моделей относительно истинного онлайн-качества.

### Proxy metrics: что предсказывает успех A/B теста

Индустриальная практика — искать **proxy metrics**: офлайн-метрики, хорошо коррелирующие с бизнес-метриками в A/B тестах. Это эмпирический процесс, требующий накопления исторических экспериментов.

| Офлайн-метрика | Связь с онлайн-метриками | Комментарий |
|---|---|---|
| NDCG (full ranking) | Средняя корреляция с CTR | Не всегда надёжно |
| Unnormalized DCG | Хорошая корреляция ([Saito et al.](https://arxiv.org/html/2307.15053v3)) | Предпочтительнее nDCG |
| Coverage (% каталога) | Слабая связь с CTR, сильная с Retention | Мера разнообразия |
| Novelty | Смешанная | Зависит от продукта |
| Calibration | Хорошая для долгосрочного Retention | Распределение рекомендаций ≈ интересам |

### Surrogate objectives

В промышленности часто обучают модели на **surrogate objective** — прокси-цели, которая вычислима офлайн и коррелирует с бизнес-метрикой. Примеры:
- Обучение на **вероятности долгого просмотра** вместо клика.
- Многозадачное обучение: одновременная оптимизация CTR + like rate + share rate.
- **Calibrated recommendations**: штраф за систематическое отклонение рекомендованного распределения жанров от исторического.

### A/B тестирование в RecSys: специфические угрозы

**Novelty effect (новинковый эффект):** пользователи взаимодействуют с новой моделью активнее просто потому, что она новая — «эффект новизны». В первые дни A/B теста метрики экспериментальной группы искусственно завышены.

*Митигация:* минимальная длительность A/B теста — 2–4 недели. Следить за **динамикой** метрик: если CTR в treatment group к концу теста возвращается к контролю, это novelty effect.

**Network effects (сетевые эффекты):** в системах с социальной компонентой (совместные плейлисты, рекомендации на основе поведения друзей) пользователи treatment и control группы не независимы — действия пользователей из treatment влияют на control.

*Митигация:* **switchback experiments** — рандомизация проводится не по пользователям, а по **временны́м периодам**: вся аудитория видит модель A, затем модель B, затем снова A и т.д.

$$
\text{ATE} = \mathbb{E}[Y_t^{\text{treat}} - Y_t^{\text{control}}], \quad t = 1, \ldots, T
$$

Switchback «переключает» систему между вариантами по расписанию (например, каждые 30 минут), что устраняет пользовательскую кластеризацию при сетевых эффектах.

## 2.8 Типичные ошибки при валидации

### Recognition features (утечка из будущего)

**Recognition features** — признаки, вычисленные с использованием информации, которая в момент предсказания недоступна.

*Примеры:*

- **Global item popularity:** посчитана по всему датасету, включая тестовый период. Модель «знает», что объект $i$ окажется популярным в будущем.
- **User-item interaction count:** общее число взаимодействий с объектом включает будущие взаимодействия.
- **Item embeddings, обученные на всём датасете:** эмбеддинги кодируют информацию о будущих взаимодействиях.
- **Нормализационные статистики (mean, std):** посчитаны на всём датасете.

**Правило:** любая статистика, входящая в признаки, должна быть вычислена **только на данных, предшествующих $t_{\text{predict}}$**.

```python
# НЕПРАВИЛЬНО: item_count считается по всему датасету
interactions_df['item_count'] = (
    interactions_df.groupby('item_id')['user_id'].transform('count')
)

# ПРАВИЛЬНО: считаем только по train
train_item_counts = train_df.groupby('item_id')['user_id'].count()
test_df['item_count'] = test_df['item_id'].map(train_item_counts).fillna(0)
```

### Nested leakage через hyperparameter tuning

Если гиперпараметры подбираются по **тестовому множеству** (явно или неявно через многократное сравнение), тестовые метрики перестают быть несмещённой оценкой.

*Скрытый вариант:* исследователь подбирает параметры вручную, запуская эксперименты на тестовых данных без явного holdout. Даже без автоматического grid search — это утечка.

*Корректная практика:*
```
Данные → Train / Val / Test
                ↑
        Tuning только здесь
                      ↑
               Финальная оценка ОДИН РАЗ
```

### Checklist: что проверить перед сдачей модели

- [ ] **Временна́я корректность сплита:** train содержит только данные до $T^*$, test — только после.
- [ ] **Фильтрация пользователей:** в тесте нет пользователей с менее чем $K_{\min}$ взаимодействиями в train.
- [ ] **Признаки без утечки:** item popularity, user statistics, embeddings — вычислены только на train.
- [ ] **Нормализация без утечки:** mean/std для нормализации признаков вычислены на train, применены к val/test.
- [ ] **Hyperparameter tuning только на val:** тест использован ровно один раз.
- [ ] **Negative sampling корректен:** негативы для оценки сэмплируются из объектов, не взаимодействовавших с пользователем в train **и** val.
- [ ] **Метрики по стратам:** приведены результаты отдельно для head / mid-tail / long-tail объектов.
- [ ] **Cold-start:** отдельно задокументировано поведение модели на новых пользователях и объектах.
- [ ] **Baseline-ы:** результаты сравнены с TopPop и другими простыми базовыми линиями.
- [ ] **Полное ранжирование:** метрики посчитаны по всему каталогу, а не по сэмплированному пулу.

> ⚠️ **Антипаттерн: тест запускается больше одного раза**
>
> Каждый запуск оценки на test при последующей корректировке модели — это неявный hyperparameter tuning на тесте. После фиксации модели тест должен запускаться строго один раз.

### Практический вывод

Офлайн-оценка — необходимый, но **недостаточный** инструмент. Архитектура валидационного пайплайна должна включать:

```
Офлайн temporal holdout
        ↓
Online A/B test (limited traffic)
        ↓
Full production rollout + мониторинг
```

Каждый уровень снижает риск деплоя некачественной модели. Офлайн-оценка отфильтровывает явно плохие кандидаты; симуляция проверяет устойчивость во времени; A/B тест даёт несмещённую оценку бизнес-метрик.

## Резюме блока

| Тема | Ключевой вывод |
|---|---|
| Random split | Нарушает временну́ю структуру, приводит к data leakage и завышенным метрикам |
| Temporal split | Global temporal — промышленный стандарт; leave-last-out — академический |
| LOO | Воспроизводим, но нереалистичен; избегайте random LOO |
| Popularity bias | Стандартные метрики смещены к head items; используйте stratified evaluation |
| MNAR / Exposure bias | Unobserved ≠ disliked; IPS и SNIPS для несмещённой оценки |
| Implicit feedback | 100-item evaluation несостоятельна (Krichene & Rendle 2020); используйте full ranking |
| Офлайн vs Онлайн | Unnormalized DCG > nDCG для корреляции с CTR; novelty effect, switchback |
| Data leakage | Recognition features, global statistics, nested tuning на тесте — основные ловушки |

---

## Ключевые ссылки

1. Krichene, W., & Rendle, S. (2022). *On Sampled Metrics for Item Recommendation.* Communications of the ACM. [DOI: 10.1145/3535335](https://dl.acm.org/doi/10.1145/3535335)
2. Campos, P.G. et al. (2020). *Exploring Data Splitting Strategies for the Evaluation of Recommendation Models.* RecSys 2020. [arXiv:2007.13237](https://arxiv.org/pdf/2007.13237)
3. Yang, L. et al. (2018). *Unbiased Offline Recommender Evaluation for Missing-Not-At-Random Implicit Feedback.* RecSys 2018. [Cornell PDF](https://www.cs.cornell.edu/~ylongqi/paper/YangCXWBE18.pdf)
4. Saito, Y. et al. (2022). *On (Normalised) Discounted Cumulative Gain as an Off-Policy Evaluation Metric for Top-n Recommendation.* [arXiv:2307.15053](https://arxiv.org/html/2307.15053v3)
5. Deffayet, R. et al. (2022). *Offline Evaluation for Reinforcement Learning-based Recommendation.* SIGIR 2023. [PDF](https://staff.fnwi.uva.nl/m.derijke/wp-content/papercite-data/pdf/deffayet-2022-offline.pdf)
6. Gao, C. et al. (2023). *KuaiSim: A Comprehensive Simulator for Recommender Systems.* NeurIPS 2023. [NeurIPS link](https://neurips.cc/virtual/2023/poster/73528)
7. Raja, K. et al. (2025). *Counterfactual Risk Minimization with IPS-Weighted BPR.* [arXiv:2509.00333](https://arxiv.org/abs/2509.00333)

# Блок 3. Learning to Rank в рекомендательных системах: функции потерь и оптимизация


## 3.1 Парадигмы Learning to Rank

### Зачем вообще нужен отдельный класс задач?

Рекомендательная система возвращает пользователю не одну точку, а **упорядоченный список** объектов. Классические метрики качества — NDCG, MAP, MRR — измеряют именно качество упорядочивания, а не точность предсказания абсолютных значений релевантности. Прямая оптимизация этих метрик нетривиальна: они, как правило, не дифференцируемы. Learning to Rank (LTR) — это семейство методов, которые обходят эту проблему, заменяя нецифференцируемую цель дифференцируемыми суррогатными функциями потерь.

Формально: пусть задан запрос (пользователь) $q$, множество кандидатов $\mathcal{D}_q = \{d_1, \ldots, d_n\}$ и истинные релевантности $\{y_1, \ldots, y_n\}$. Модель $f_\theta$ выдаёт скоры $s_i = f_\theta(q, d_i)$, по которым строится ранжирование $\pi$. Нам нужно обучить $\theta$ так, чтобы $\pi$ был близок к оптимальному.

Все методы LTR делятся на три парадигмы в зависимости от того, **на какой единице обучения** строится функция потерь.

---

### 3.1.1 Pointwise

**Идея.** Рассматриваем каждый документ независимо. Задача сводится к регрессии или классификации: предсказать $y_i$ по $(q, d_i)$.

$$
\mathcal{L}_{\text{pointwise}} = \sum_{i=1}^{n} \ell(f_\theta(q, d_i),\, y_i)
$$

Например, $\ell = \text{MSE}$ для регрессии или $\ell = \text{BCE}$ для бинарных меток.

**Проблемы:**
- Модель не видит контекст: она не знает, что $d_3$ лучше $d_7$ — ей сообщают только абсолютные метки.
- Ошибка в абсолютных значениях скора может не влиять на порядок (и наоборот). Пример: если истинные релевантности $[3, 2, 1]$, предсказания $[10, 5, 0]$ и $[0.3, 0.2, 0.1]$ дают одинаковое ранжирование, но разный MSE.
- Несбалансированность: нерелевантных объектов обычно на порядки больше, чем релевантных.

---

### 3.1.2 Pairwise

**Идея.** Обучаемся на **парах** $(d_i, d_j)$ с условием $y_i > y_j$. Функция потерь штрафует за нарушение порядка.

$$
\mathcal{L}_{\text{pairwise}} = \sum_{(i,j):\, y_i > y_j} \ell(f_\theta(q, d_i) - f_\theta(q, d_j))
$$

Примеры: RankNet, RankSVM, IR-SVM. Pairwise-подходы естественно кодируют задачу ранжирования, однако:
- Количество пар $O(n^2)$ — вычислительно дорого для больших списков.
- Все пары считаются равнозначными, хотя перестановка на первой позиции важнее, чем на последней.

---

### 3.1.3 Listwise

**Идея.** Функция потерь определена на **всём списке** сразу. Можно непосредственно аппроксимировать NDCG, MAP или оптимизировать вероятность правильной перестановки.

$$
\mathcal{L}_{\text{listwise}} = \mathcal{L}(f_\theta(q, d_1), \ldots, f_\theta(q, d_n),\; y_1, \ldots, y_n)
$$

Примеры: ListNet, ListMLE, SoftRank, ApproxNDCG. Listwise-методы концептуально ближе к цели, но сложнее в реализации и оптимизации.

---

### 3.1.4 Сравнительная таблица

| Критерий | Pointwise | Pairwise | Listwise |
|---|---|---|---|
| Единица обучения | Один объект | Пара объектов | Весь список |
| Сложность обучения | $O(n)$ | $O(n^2)$ пар | $O(n)$–$O(n^2)$ |
| Близость к метрике | Низкая | Средняя | Высокая |
| Чувствительность к позиции | Нет | Частично (LambdaRank) | Да |
| Простота реализации | Высокая | Средняя | Низкая |
| Примеры методов | Linear regression, GBDT | RankNet, BPR, SVM-Rank | ListNet, ListMLE, SoftRank |

## 3.2 Pairwise подходы: RankNet

### Постановка

RankNet (Burges et al., 2005) — один из первых нейросетевых LTR-методов, разработанный в Microsoft для веб-поиска. Ключевая идея: моделировать **вероятность** того, что документ $d_i$ должен стоять выше $d_j$.

Пусть модель выдаёт скоры $s_i = f_\theta(q, d_i)$ и $s_j = f_\theta(q, d_j)$. Определим вероятность правильного упорядочивания через сигмоиду от разности скоров:

$$
\boxed{P(d_i \succ d_j) = \sigma(s_i - s_j) = \frac{1}{1 + e^{-(s_i - s_j)}}}
$$

Обозначим целевую вероятность:

$$
\bar{P}_{ij} = \begin{cases} 1 & \text{если } y_i > y_j \\ 0.5 & \text{если } y_i = y_j \\ 0 & \text{если } y_i < y_j \end{cases}
$$

### Функция потерь

RankNet минимизирует бинарную кросс-энтропию между предсказанной и целевой вероятностью:

$$
\mathcal{L}_{\text{RankNet}} = -\sum_{(i,j):\, y_i > y_j} \left[ \bar{P}_{ij} \log P_{ij} + (1 - \bar{P}_{ij}) \log (1 - P_{ij}) \right]
$$

Для случая $\bar{P}_{ij} = 1$ (строгий порядок $y_i > y_j$):

$$
\mathcal{L}_{ij} = -\log \sigma(s_i - s_j) = \log\left(1 + e^{-(s_i - s_j)}\right)
$$

### Градиент и его интерпретация

Вычислим градиент потери по скору $s_i$:

$$
\frac{\partial \mathcal{L}_{ij}}{\partial s_i} = -\frac{1}{1 + e^{s_i - s_j}} \cdot \left(-1\right) \cdot \ldots
$$

Точнее, раскроем аккуратно. Обозначим $o_{ij} = s_i - s_j$:

$$
\mathcal{L}_{ij} = \log(1 + e^{-o_{ij}})
$$

$$
\frac{\partial \mathcal{L}_{ij}}{\partial s_i} = \frac{-e^{-o_{ij}}}{1 + e^{-o_{ij}}} = -(1 - \sigma(o_{ij})) = \sigma(o_{ij}) - 1 = P_{ij} - 1
$$

$$
\frac{\partial \mathcal{L}_{ij}}{\partial s_j} = 1 - P_{ij}
$$

**Интерпретация:** если $P_{ij}$ близко к 1 (модель уверена в правильном порядке), градиент мал. Если $P_{ij} \approx 0$ (модель ошибается), градиент велик — модель сильно штрафуется. Это аналог поведения логистической регрессии.

Итоговый градиент по параметрам $\theta$:

$$
\frac{\partial \mathcal{L}}{\partial \theta} = \sum_{(i,j):\, y_i > y_j} (P_{ij} - 1) \frac{\partial s_i}{\partial \theta} + (1 - P_{ij}) \frac{\partial s_j}{\partial \theta}
$$

### Ограничения RankNet

1. Все пары $(i, j)$ с $y_i > y_j$ считаются **равнозначными**. Перестановка объектов на позициях 1 и 2 столь же важна (с точки зрения потери), что и перестановка на позициях 100 и 101. Реальные метрики типа NDCG или MRR экспоненциально чувствительнее к верхним позициям.
2. Градиент не учитывает **контекст всего списка**.

## 3.3 LambdaRank и LambdaMART

### Проблема: NDCG не дифференцируема

Normalized Discounted Cumulative Gain:

$$
\text{NDCG}@K = \frac{\text{DCG}@K}{\text{IDCG}@K}, \quad \text{DCG}@K = \sum_{k=1}^{K} \frac{2^{y_{\pi(k)}} - 1}{\log_2(k+1)}
$$

где $\pi(k)$ — документ на позиции $k$. Ранжирование — кусочно-постоянная функция от скоров: при малом изменении $s_i$ порядок не меняется (и градиент равен нулю), а при пересечении скоров двух документов NDCG может резко измениться (градиент разрывен). **Прямая оптимизация NDCG через обратное распространение ошибки невозможна**.

### LambdaRank: виртуальные градиенты

Burges et al. (2006) предложили хитрый трюк: **не определять функцию потерь**, а напрямую задать **градиенты** по скорам как виртуальные сигналы. Это обходит необходимость существования дифференцируемой функции потерь.

Для пары $(i, j)$ с $y_i > y_j$ определим $\lambda$-градиент:

$$
\boxed{\lambda_{ij} = \frac{\partial \mathcal{L}_{ij}}{\partial s_i} \cdot |\Delta\text{NDCG}_{ij}|}
$$

где $\frac{\partial \mathcal{L}_{ij}}{\partial s_i} = \sigma(o_{ij}) - 1$ — градиент RankNet-потери, а $|\Delta\text{NDCG}_{ij}|$ — **абсолютное изменение NDCG**, которое произошло бы при обмене позиций $d_i$ и $d_j$ в текущем ранжировании.

Полный $\lambda$-градиент для документа $i$ по всему списку:

$$
\Lambda_i = \sum_{j:\, y_j < y_i} \lambda_{ij} - \sum_{j:\, y_j > y_i} \lambda_{ji}
$$

Вычисление $|\Delta\text{NDCG}_{ij}|$: пусть $d_i$ стоит на позиции $r_i$, $d_j$ — на $r_j$ ($r_i < r_j$, т.е. $d_i$ выше). При обмене позиций:

$$
|\Delta\text{DCG}_{ij}| = (2^{y_i} - 2^{y_j}) \left| \frac{1}{\log_2(r_i+1)} - \frac{1}{\log_2(r_j+1)} \right|
$$

$$
|\Delta\text{NDCG}_{ij}| = \frac{|\Delta\text{DCG}_{ij}|}{\text{IDCG}}
$$

### Интуиция LambdaRank

Классическая аналогия: представьте, что вы тренируете прыгуна в длину. Вы не можете дать ему "функцию длины прыжка" — он уже прыгает как умеет. Но вы можете **указывать направление**: "двигай бедра вперёд сильнее" или "слабее". LambdaRank делает ровно это: он не говорит модели "твоя потеря равна X", но сообщает "скор объекта $i$ нужно поднять сильнее, если его подъём на одну позицию улучшит NDCG на 0.1, и слабее, если — на 0.001".

Ключевое свойство: пары в топе списка автоматически получают **больший вес**, так как $|\Delta\text{NDCG}|$ для перестановок в начале выше из-за логарифмического дисконта.

### LambdaMART

LambdaMART = **Lambda**Rank + **MART** (Multiple Additive Regression Trees, синоним GBDT).

GBDT строит ансамбль деревьев $F_T(x) = \sum_{t=1}^T \eta \cdot h_t(x)$, где каждое новое дерево $h_t$ аппроксимирует **псевдо-остатки** — то есть градиент функции потерь по предсказаниям. В LambdaMART роль псевдо-остатков играют $\Lambda_i$:

$$
h_t(x_i) \approx -\Lambda_i \Big|_{F = F_{t-1}}
$$

**Почему GBDT хорошо работает в промышленности для LTR:**

1. **Устойчивость к выбросам и пропускам.** Деревья не требуют нормализации признаков.
2. **Эффективная обработка гетерогенных признаков.** На практике признаки запроса и документа имеют разные масштабы (TF-IDF, категории, счётчики).
3. **Интерпретируемость.** Feature importance легко вычислима.
4. **Вычислительная эффективность.** Инкрементальное добавление деревьев, ранняя остановка.
5. **Качество.** LambdaMART долго оставался state-of-the-art на стандартных LTR-бенчмарках (LETOR, Yahoo! Learning to Rank).

```python
# Псевдокод LambdaMART
def lambdaMART(queries, docs, labels, T=100, lr=0.1):
    F = zeros(len(docs))  # начальные скоры
    for t in range(T):
        lambdas = compute_lambda_gradients(F, labels)  # ΔΛ_i по формуле выше
        hessians = compute_hessians(F, labels)         # вторые производные
        tree = fit_regression_tree(docs, -lambdas, hessians)
        leaf_vals = optimize_leaf_values(tree, lambdas, hessians)
        F += lr * tree.predict(docs)
    return F
```

## 3.4 Triplet Loss

### Постановка задачи

Triplet Loss широко используется в метрическом обучении и адаптируется для задач ранжирования в RecSys при обучении эмбеддингов. Задача: научить пространство эмбеддингов так, чтобы **похожие объекты были близко, непохожие — далеко**.

Каждый обучающий пример — тройка:
- **Anchor** $a$: опорный объект (например, пользователь или запрос);
- **Positive** $p$: объект, релевантный анкору ($y_{ap} = 1$);
- **Negative** $n$: нерелевантный объект ($y_{an} = 0$).

### Формула Triplet Loss

$$
\boxed{\mathcal{L}_{\text{triplet}} = \sum_{(a,p,n)} \max\left(0,\; d(a, p) - d(a, n) + \alpha\right)}
$$

где $d(\cdot, \cdot)$ — расстояние (евклидово или косинусное), $\alpha > 0$ — **margin** (зазор).

**Интерпретация:** потеря равна нулю, когда негатив дальше от анкора, чем позитив, **с запасом** $\alpha$. Это важно: без margin сеть может коллапсировать к нулевым эмбеддингам.

Используя евклидово расстояние $d(u, v) = \|u - v\|_2$:

$$
\mathcal{L} = \max\left(0,\; \|f(a) - f(p)\|_2 - \|f(a) - f(n)\|_2 + \alpha\right)
$$

### Offline vs Online Triplet Mining

**Offline mining:** тройки формируются заранее до начала обучения.
- Простота, но фиксированный набор тяжёлых примеров быстро устаревает по мере обучения модели.
- Риск: тройки, которые были "сложными" на старте, становятся тривиальными через несколько эпох.

**Online mining:** тройки формируются **внутри каждого батча** $\mathcal{B} = \{(x_k, y_k)\}_{k=1}^B$.
- Актуальность: тройки пересчитываются по текущим эмбеддингам.
- Эффективность: за один forward pass по батчу можно получить $O(B^3)$ тройк (в батче из $B$ объектов).

### Hard, Semi-hard и Easy Negatives

Ключевая проблема triplet loss — **качество Mining'а негативов**.

Классификация негативов для пары $(a, p)$:

| Тип негатива | Условие | Поведение потери |
|---|---|---|
| **Easy** | $$d(a,n) > d(a,p) + \alpha$$ | $\mathcal{L} = 0$, обучения нет |
| **Semi-hard** | $$d(a,p) < d(a,n) < d(a,p) + \alpha$$ | $\mathcal{L} > 0$, стабильное обучение |
| **Hard** | $$d(a,n) < d(a,p)$$ | $\mathcal{L} > \alpha$, агрессивное обучение |

**Проблема easy negatives — коллапс:**
Если большинство тройк — easy, то почти все слагаемые потери равны нулю. Градиенты не поступают, обучение стагнирует. Хуже того: модель может коллапсировать в вырожденное решение (например, все эмбеддинги = $\vec{0}$), т.к. при этом $d(a,p) = d(a,n) = 0$ и потеря = max(0, 0 - 0 + α) = α, что не является нулём — но сеть может попасть в такой минимум при плохой инициализации.

**Проблема hard negatives — нестабильность:**
Чисто hard mining приводит к нестабильному обучению: в начале обучения, когда эмбеддинги случайны, почти все тройки "тяжёлые", что даёт огромные градиенты и взрыв потери.

**Semi-hard — компромисс:**
Schroff et al. (FaceNet, 2015) предложили брать негативы, которые **нарушают условие margin, но ещё не hard**: $d(a,p) < d(a,n) < d(a,p) + \alpha$. Это обеспечивает ненулевой градиент без нестабильности.

### Стратегии BatchAll и BatchHard

Обе стратегии — варианты online mining на уровне батча.

**BatchAll:**
- Берём **все** валидные тройки $(a, p, n)$ из батча.
- Вычисляем потерю по каждой, усредняем ненулевые.

$$
\mathcal{L}_{\text{BatchAll}} = \frac{1}{|\mathcal{T}_{+}|} \sum_{(a,p,n) \in \mathcal{T}_{+}} \max(0, d_{ap} - d_{an} + \alpha)
$$

где $\mathcal{T}_{+}$ — тройки с ненулевой потерей.

- Плюс: полная информация из батча.
- Минус: многие easy пары разбавляют сигнал.

**BatchHard:**
Для каждого анкора $a$ выбираем:
- **Hardest positive:** $p^* = \arg\max_p d(a, p)$ — позитив, наиболее далёкий от анкора.
- **Hardest negative:** $n^* = \arg\min_n d(a, n)$ — негатив, наиболее близкий к анкору.

$$
\mathcal{L}_{\text{BatchHard}} = \frac{1}{|\mathcal{A}|} \sum_{a \in \mathcal{A}} \max\left(0,\; \max_p d(a, p) - \min_n d(a, n) + \alpha\right)
$$

- Плюс: фокус на самых информативных примерах, хорошая сходимость.
- Минус: на ранних эпохах может быть нестабильным.

```python
def batch_hard_triplet_loss(embeddings, labels, margin=0.3):
    """
    embeddings: (B, D) — эмбеддинги батча
    labels:     (B,)   — классы/id объектов
    """
    # Попарные расстояния (B x B)
    dist = pairwise_euclidean(embeddings)

    # Маски: positive = same label, negative = diff label
    pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1))  # (B, B)
    neg_mask = ~pos_mask

    # Hardest positive per anchor: max distance среди позитивов
    hardest_pos = (dist * pos_mask.float()).max(dim=1).values

    # Hardest negative per anchor: min distance среди негативов
    # Заполняем позитивные позиции inf, чтобы не взять их как min
    dist_neg = dist.masked_fill(pos_mask, float('inf'))
    hardest_neg = dist_neg.min(dim=1).values

    loss = F.relu(hardest_pos - hardest_neg + margin)
    return loss.mean()
```

## 3.5 InfoNCE и контрастивное обучение

### Контрастивное обучение в рекомендациях

Контрастивное обучение (Contrastive Learning) обучает модель максимизировать схожесть между **положительной парой** и минимизировать схожесть с **негативами**. В RecSys это означает: пользователь $u$ должен быть близок к товарам, с которыми он взаимодействовал (позитивы), и далёк от остальных (негативы).

**In-batch negatives** — практически важный трюк: все остальные элементы батча используются как негативы. Если батч содержит $B$ пар $(u_k, d_k^+)$, то для пользователя $u_i$ негативами служат $\{d_j^+\}_{j \neq i}$.

### InfoNCE Loss

InfoNCE (Оozma et al. / van den Oord et al., 2018):

$$
\boxed{\mathcal{L}_{\text{InfoNCE}} = -\frac{1}{B} \sum_{i=1}^{B} \log \frac{\exp\left(\text{sim}(u_i, d_i^+) / \tau\right)}{\sum_{j=1}^{B} \exp\left(\text{sim}(u_i, d_j) / \tau\right)}}
$$

где $\text{sim}(u, d) = \frac{u \cdot d}{\|u\| \|d\|}$ — косинусное сходство, $\tau > 0$ — **температурный параметр**.

Это в точности кросс-энтропия по $B$ классам, где правильный класс — индекс $i$.

### Роль температурного параметра $\tau$

Температура $\tau$ контролирует **остроту распределения** вероятностей:

$$
p_j = \frac{\exp(\text{sim}(u_i, d_j) / \tau)}{\sum_k \exp(\text{sim}(u_i, d_k) / \tau)}
$$

- **Малое $\tau$** (например, 0.05): распределение острее, почти всё внимание сосредоточено на ближайших негативах. Обучение сфокусировано на hard negatives — хорошая сходимость, но риск нестабильности.
- **Большое $\tau$** (например, 1.0): распределение равномернее, мягкие градиенты по всем негативам. Медленная, но стабильная сходимость.

Интуиция через энтропию: при $\tau \to 0$ задача вырождается в жёсткую классификацию (winner-takes-all), при $\tau \to \infty$ — в равномерное распределение без обучающего сигнала.

**Типичное значение** в практических системах: $\tau \in [0.05, 0.2]$ (SimCLR использует 0.07, YouTube DNN-подобные системы — около 0.05–0.1).

### Связь InfoNCE с взаимной информацией

InfoNCE является нижней оценкой взаимной информации $I(U; D)$:

$$
\mathcal{L}_{\text{InfoNCE}} \geq -I(U; D) + \log B
$$

Таким образом, минимизация InfoNCE эквивалентна максимизации нижней оценки взаимной информации между пользователями и документами. Это теоретическое обоснование объясняет, почему метод хорошо обобщается.

### SimCLR → RecSys: адаптация

SimCLR (Chen et al., 2020) изначально разработан для CV: аугментации изображения образуют позитивную пару, негативы — остальные изображения в батче.

В RecSys аналогия следующая:

| SimCLR (CV) | RecSys-адаптация |
|---|---|
| Два вида аугментации одного изображения | Два разных окна взаимодействий одного пользователя |
| Позитивная пара: два вида одного объекта | Позитивная пара: (пользователь, взаимодействовавший товар) |
| Негативы: другие изображения в батче | Негативы: товары других пользователей в батче |
| Проекционная голова $g(\cdot)$ | MLP поверх user/item-эмбеддингов |

```python
def info_nce_loss(user_emb, item_emb, tau=0.1):
    """
    user_emb: (B, D) — эмбеддинги пользователей
    item_emb: (B, D) — эмбеддинги соответствующих позитивных айтемов
    """
    # L2-нормировка
    u = F.normalize(user_emb, dim=-1)  # (B, D)
    d = F.normalize(item_emb, dim=-1)  # (B, D)

    # Матрица сходств: (B, B), где [i, j] = sim(u_i, d_j)
    logits = torch.matmul(u, d.T) / tau  # (B, B)

    # Метки: каждый пользователь i соответствует айтему i
    labels = torch.arange(B)

    # Cross-entropy: строки — пользователи, столбцы — айтемы
    return F.cross_entropy(logits, labels)
```

## 3.6 BPR: Bayesian Personalized Ranking

### Постановка: имплицитные данные

В большинстве реальных RecSys данные **имплицитны**: у нас есть информация о взаимодействиях (просмотр, клик, покупка), но не явные оценки. Из факта взаимодействия пользователя $u$ с товаром $i$ мы знаем только, что $i$ предпочтительнее, чем случайный товар $j$, с которым $u$ не взаимодействовал.

Ренде и Фридрих (Rendle et al., 2009) сформулировали BPR как **pairwise байесовский критерий**:

$$
u \text{ предпочитает } i \succ j, \text{ если } i \in \mathcal{I}_u^+, \; j \notin \mathcal{I}_u^+
$$

где $\mathcal{I}_u^+ = \{i : (u, i) \in \mathcal{O}^+\}$ — множество товаров, с которыми $u$ взаимодействовал.

### Вывод BPR: MAP-постановка

Нам нужно найти параметры $\Theta$, максимизирующие апостериорную вероятность:

$$
p(\Theta \mid >_u) \propto p(>_u \mid \Theta) \cdot p(\Theta)
$$

Предполагаем, что предпочтения по парам независимы:

$$
p(>_u \mid \Theta) = \prod_{(u,i,j) \in \mathcal{D}_S} p(i >_u j \mid \Theta)
$$

где $\mathcal{D}_S = \{(u, i, j) : i \in \mathcal{I}_u^+, j \notin \mathcal{I}_u^+\}$.

Вероятность предпочтения моделируется через разность скоров:

$$
p(i >_u j \mid \Theta) = \sigma(\hat{x}_{uij}), \quad \hat{x}_{uij} = \hat{x}_{ui} - \hat{x}_{uj}
$$

где $\hat{x}_{ui}$ — скор модели (например, матричного разложения $\hat{x}_{ui} = \mathbf{u}^\top \mathbf{v}_i$).

С гауссовским приором $p(\Theta) = \mathcal{N}(0, \lambda_\Theta I)$, логарифм апостериорной вероятности:

$$
\ln p(\Theta \mid >_u) = \sum_{(u,i,j) \in \mathcal{D}_S} \ln \sigma(\hat{x}_{uij}) - \lambda_\Theta \|\Theta\|^2
$$

### BPR Loss

Переходим к минимизируемой функции потерь:

$$
\boxed{\mathcal{L}_{\text{BPR}} = -\sum_{(u,i,j) \in \mathcal{D}_S} \ln \sigma(\hat{x}_{ui} - \hat{x}_{uj}) + \lambda_\Theta \|\Theta\|^2}
$$

### Алгоритм оптимизации: LearnBPR

BPR-Opt оптимизируется стохастическим градиентным спуском с равномерной выборкой троек:

```
LearnBPR(данные D_S, параметры Θ):
  инициализировать Θ случайно
  повторять:
    (u, i, j) ← случайная тройка из D_S
    x_uij ← x_ui(Θ) - x_uj(Θ)
    e ← 1 - σ(x_uij)          # ошибка
    Θ ← Θ + α * (e * ∂x_uij/∂Θ - λ * Θ)
  до сходимости
```

### BPR vs. Cross-Entropy для implicit feedback

| Критерий | BPR | Binary Cross-Entropy |
|---|---|---|
| Тип | Pairwise | Pointwise |
| Метка | $(u, i) \succ (u, j)$ | $y_{ui} \in \{0, 1\}$ |
| Интерпретация нулей | Ненаблюдаемые = негативы в парах | Ненаблюдаемые = жёсткие 0 |
| Bias при дисбалансе | Устойчив (парное сравнение) | Сильный (много нулей доминируют) |
| Negative sampling | Обязателен (тройки) | Обязателен (отрицательные примеры) |
| Близость к метрике | Близко к AUC | Далеко от NDCG/MAP |
| Сложность на шаг | $O(1)$ (одна тройка) | $O(|\mathcal{I}|)$ без sampling |

**Ключевое преимущество BPR:** метод явно обучается на **попарном порядке**, что соответствует задаче ранжирования. BCE трактует каждое взаимодействие изолированно и страдает от сильного дисбаланса классов (99%+ нулей в матрице взаимодействий).

## 3.7 Listwise подходы

### Мотивация

Listwise-методы оптимизируют функцию потерь сразу по **всему ранжированному списку**, что концептуально ближе к целевым метрикам. Ключевой математический инструмент — **вероятностные модели перестановок**, прежде всего распределение Plackett-Luce.

### Распределение Plackett-Luce

Вероятность конкретной перестановки $\pi = (\pi_1, \pi_2, \ldots, \pi_n)$ при скорах $s = (s_1, \ldots, s_n)$:

$$
P(\pi \mid s) = \prod_{k=1}^{n} \frac{\exp(s_{\pi_k})}{\sum_{j=k}^{n} \exp(s_{\pi_j})}
$$

Это произведение softmax'ов: на первой позиции стоит документ с вероятностью $\propto \exp(s_{\pi_1})$ из всех $n$; на второй — из оставшихся $n-1$; и т.д. Marginal-вероятность того, что документ $i$ стоит на позиции $k$, не имеет замкнутой формы для больших $n$, что является главной вычислительной трудностью listwise-методов.

### 3.7.1 ListNet

Cao et al. (2007) предложили ListNet, основанный на минимизации KL-дивергенции между распределением перестановок, заданным предсказанными скорами, и идеальным распределением по меткам релевантности.

Для практического использования авторы предлагают **top-one probability** — вероятность того, что документ $i$ стоит на первой позиции:

$$
P_s(i) = \frac{\exp(s_i)}{\sum_{k=1}^{n} \exp(s_k)}, \quad P_y(i) = \frac{\exp(y_i)}{\sum_{k=1}^{n} \exp(y_k)}
$$

**ListNet Loss:**

$$
\mathcal{L}_{\text{ListNet}} = -\sum_{i=1}^{n} P_y(i) \log P_s(i) = H(P_y, P_s)
$$

Это кросс-энтропия между "идеальным" softmax по меткам и предсказанным softmax по скорам. При этом SoftMax loss является частным случаем ListNet при бинарных метках (один позитив, остальные негативы):

$$
P_y(i^+) = 1, \quad \mathcal{L} = -\log P_s(i^+) = -\log \frac{\exp(s_{i^+})}{\sum_k \exp(s_k)}
$$

Это в точности стандартный softmax cross-entropy loss, широко применяемый в two-tower моделях.

**Градиент ListNet:**

$$
\frac{\partial \mathcal{L}_{\text{ListNet}}}{\partial s_i} = -(P_y(i) - P_s(i))
$$

Простая и элегантная формула: градиент пропорционален разности между идеальным и предсказанным распределениями.

### 3.7.2 ListMLE

Shi et al. (2010) предложили ListMLE как максимизацию правдоподобия **оптимальной перестановки** $\pi^*$ (ранжирование по убыванию $y_i$):

$$
\mathcal{L}_{\text{ListMLE}} = -\log P(\pi^* \mid s) = -\sum_{k=1}^{n} \left[ s_{\pi^*_k} - \log \sum_{j=k}^{n} \exp(s_{\pi^*_j}) \right]
$$

Раскроем:

$$
\mathcal{L}_{\text{ListMLE}} = \sum_{k=1}^{n} \log\left(1 + \sum_{j=k+1}^{n} \exp(s_{\pi^*_j} - s_{\pi^*_k})\right)
$$

**Интерпретация:** для каждой позиции $k$ в оптимальном ранжировании стоимость равна логарифму нормировочного коэффициента относительно всех документов ниже по списку. Модель штрафуется за то, что документы с низкой релевантностью имеют высокий скор.

**Сравнение ListNet и ListMLE:**

| | ListNet | ListMLE |
|---|---|---|
| Оптимизируемый объект | KL-дивергенция softmax-распределений | Log-likelihood оптимальной перестановки |
| Computational cost | $O(n)$ за список | $O(n^2)$ в наивной реализации, $O(n \log n)$ |
| Учитывает весь порядок | Нет (top-one approx.) | Да |
| Чувствительность к позициям | Нет | Нет (нет явного дисконта) |

## 3.8 Стратегии Negative Sampling

### Проблема

В RecSys типичное соотношение позитивных и негативных объектов — $1 : 10^4$ и хуже. Использовать все возможные негативы при каждом обновлении параметров вычислительно невозможно. Необходима **выборка негативов**.

### Uniform sampling

Каждый негативный объект выбирается равномерно: $P_{\text{uniform}}(j) = \frac{1}{|\mathcal{I} \setminus \mathcal{I}_u^+|}$.

- Просто, но неэффективно: большинство случайных негативов — лёгкие.
- Не учитывает распределение популярности.

### Popularity-based sampling

Выборка пропорционально популярности айтема $c_j$ (количество взаимодействий):

$$
P_{\text{pop}}(j) \propto c_j^\alpha, \quad \alpha \in (0, 1]
$$

При $\alpha = 1$ — строго пропорционально популярности. На практике часто используют $\alpha = 0.75$ (аналог Word2Vec negative sampling).

**Логика:** популярные айтемы встречаются в тренировочном корпусе чаще, поэтому модель обучается лучше им противостоять.

### Hard negative sampling

Выбор негативов, максимально похожих на пользователя по текущей модели:

$$
n^* = \arg\max_{j \notin \mathcal{I}_u^+} \hat{x}_{uj}
$$

**Эффективнее** для обучения, но требует постоянного обновления списка кандидатов (часто с помощью ANN-индексов типа FAISS).

**Риск:** обучение на слишком hard негативах может создавать **false negatives** — ненаблюдаемые позитивы, которые оказались высоко ранжированными, но не попали в обучающую выборку.

### In-batch negatives и Popularity Bias

При использовании **in-batch negatives** (как в InfoNCE) негативами для пользователя $u_i$ служат айтемы $d_j, j \neq i$ из того же батча.

**Проблема popularity bias:** если батч формируется случайно из лога взаимодействий, популярные айтемы будут встречаться в батче чаще. Это означает, что они чаще будут использоваться как негативы → модель усиленно учится отталкиваться от популярных айтемов → при инференсе предпочтёт нишевые айтемы. **Это противоположно желаемому поведению**.

### LogQ Correction

Yi et al. (Google, 2019) и аналогичные работы предложили корректировать скоры на **логарифм вероятности выборки**:

$$
\text{sim}_{\text{corrected}}(u_i, d_j) = \text{sim}(u_i, d_j) - \log Q(d_j)
$$

где $Q(d_j)$ — вероятность попадания айтема $d_j$ в батч как негатив (пропорциональна частоте айтема в данных).

**Итоговая InfoNCE с logQ-коррекцией:**

$$
\mathcal{L}_{\text{corr}} = -\frac{1}{B} \sum_{i=1}^{B} \log \frac{\exp\left((\text{sim}(u_i, d_i^+) - \log Q(d_i^+)) / \tau\right)}{\sum_{j=1}^{B} \exp\left((\text{sim}(u_i, d_j) - \log Q(d_j)) / \tau\right)}
$$

**Интуиция:** вычитание $\log Q(d_j)$ де-факто "штрафует" популярные айтемы-негативы за то, что они слишком часто встречаются. Это эквивалентно ресэмплингу негативов в сторону равномерного распределения.

```python
def info_nce_with_logq(user_emb, item_emb, item_freq, tau=0.1):
    """
    item_freq: (B,) — частота каждого айтема в обучающих данных
    """
    u = F.normalize(user_emb, dim=-1)
    d = F.normalize(item_emb, dim=-1)

    logits = torch.matmul(u, d.T) / tau  # (B, B)

    # logQ-коррекция: нормируем частоты, берём log
    Q = item_freq / item_freq.sum()
    log_q = torch.log(Q + 1e-8).unsqueeze(0)  # (1, B)

    # Вычитаем logQ из logits
    logits_corrected = logits - log_q

    labels = torch.arange(len(u))
    return F.cross_entropy(logits_corrected, labels)
```

### Сводная таблица стратегий выборки негативов

| Стратегия | Качество сигнала | Вычислительная сложность | Риски |
|---|---|---|---|
| Uniform | Низкое (easy negatives) | $O(1)$ | Медленная сходимость |
| Popularity-based | Среднее | $O(1)$ | Popularity bias |
| Hard (offline) | Высокое | $O(n)$ периодически | False negatives, устаревание |
| Hard (online/FAISS) | Очень высокое | $O(\log n)$ per step | False negatives |
| In-batch | Среднее–высокое | $O(B^2)$ | Popularity bias |
| In-batch + logQ | Высокое, несмещённое | $O(B^2)$ | Требует оценки частот |

---

## Резюме блока

| Метод | Парадигма | Ключевая идея | Применение |
|---|---|---|---|
| RankNet | Pairwise | $P(i \succ j) = \sigma(s_i - s_j)$, BCE | Веб-поиск, LTR |
| LambdaRank | Pairwise+ | Виртуальные $\lambda$-градиенты, взвешенные по $|\Delta\text{NDCG}|$ | Поиск, ранжирование |
| LambdaMART | Pairwise+ | LambdaRank + GBDT | Промышленные системы |
| Triplet Loss | Pairwise | $\max(0, d_{ap} - d_{an} + \alpha)$, mining | Эмбеддинги, RecSys |
| InfoNCE | Listwise | Softmax по in-batch негативам с температурой $\tau$ | Two-tower, CL |
| BPR | Pairwise | MAP-оценка парного порядка для implicit | CF-модели |
| ListNet | Listwise | KL между softmax-распределениями | LTR |
| ListMLE | Listwise | MLE оптимальной перестановки (Plackett-Luce) | LTR |

**Центральный принцип блока:** ни одна из рассмотренных функций потерь не оптимизирует NDCG или MAP напрямую. Все они являются **дифференцируемыми суррогатами**, связанными с целевой метрикой либо математически (как ListNet — через вероятность перестановок), либо эвристически (как LambdaRank — через взвешивание градиентов). Выбор суррогата определяет компромисс между вычислительной стоимостью, близостью к метрике и стабильностью оптимизации.